# 🧪 TC vs Herschel–Bulkley on a flow curve — the `rheofit` way

*The original version of this notebook built its own TC and Herschel–Bulkley
models in `lmfit` and tidied the results with `pybroom`. This rewrite fits the
same `Flow_curve_example.xls` data with the `rheofit` 1.x API: pick a model key,
fit, plot.*

**Run each cell in order** (▶). The first cell installs `rheofit` in your browser — no server involved.

In [ ]:
%pip install -q rheofit rheopy-rheodata xlrd

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import rheofit
import rheodata
print("rheofit", rheofit.__version__ if hasattr(rheofit, "__version__") else "installed")
print("rheodata", getattr(rheodata, "__version__", "installed"))

## 📂 The data

"Flow sweep – 2" of `Flow_curve_example.xls` (41 points, 100 → 0.01 s⁻¹), which
ships next to this notebook. (Not curated in `rheodata`, so the notebook keeps
reading the bundled file.)

In [ ]:
XLS = next(p for p in [Path("Flow_curve_example.xls"),
                       Path("content/flow_curves_fit/Flow_curve_example.xls")]
           if p.exists())
print("using:", XLS)

raw = pd.ExcelFile(XLS, engine="xlrd").parse("Flow sweep - 2",
                                             skiprows=[0, 2])[["Shear rate", "Stress"]].dropna()
df = pd.DataFrame({"Shear rate / 1/s": raw["Shear rate"].to_numpy(),
                   "Stress / Pa": raw["Stress"].to_numpy()})
df = df.sort_values("Shear rate / 1/s").reset_index(drop=True)
print(f"{len(df)} points, "
      f"shear rate {df['Shear rate / 1/s'].min():.3g}–{df['Shear rate / 1/s'].max():.3g} 1/s, "
      f"stress {df['Stress / Pa'].min():.3g}–{df['Stress / Pa'].max():.3g} Pa")
df.head()

## 📐 Two models, one line each

The **three-component model** `tc`

σ = σ_y + σ_y·(γ̇/γ̇_c)^0.5 + η_bg·γ̇

against the **Herschel–Bulkley model** `herschel_bulkley`

σ = σ_y + K·γ̇ⁿ

— the same comparison the original notebook made, now as two model keys:

In [ ]:
res_tc = rheofit.fit(df, "tc", effort="thorough", seed=0)
res_hb = rheofit.fit(df, "herschel_bulkley", effort="normal", seed=0)

def show(res):
    print("RedChi2 = %.3e, cond = %s" % (res["redchi"], res["cond"]))
    for k, v in res["params"].items():
        err = v["stderr"] / v["value"] * 100 if v["value"] else float("nan")
        print("  %-12s = %10.4g ± %.1f%%" % (k, v["value"], err))

print("TC:")
show(res_tc)
print("HB:")
show(res_hb)
print("RedChi2(HB) / RedChi2(TC) = %.0f" % (res_hb["redchi"] / res_tc["redchi"]))

**Expect:** TC — σ_y ≈ 0.01632 Pa (±0.3 %), γ̇_c ≈ 0.1435 s⁻¹ (±1.3 %),
η_bg ≈ 0.003305 Pa·s (±1.0 %), RedChi² ≈ 3.67×10⁻⁵, cond ≈ 15.5 — a clean,
fully identified fit. HB — σ_y ≈ 0.0188 Pa, K ≈ 0.0432 Pa·sⁿ, n ≈ 0.603,
RedChi² ≈ 1.08×10⁻³ — about **29× worse**. On this dataset the TC model's
fixed-½ plastic term is exactly right, and the free exponent buys HB nothing.

In [ ]:
rheofit.plot({"TC": df, "HB": df}, fits={"TC": res_tc, "HB": res_hb},
             test_type="flow_curve", title="TC vs Herschel–Bulkley")
plt.show()

The residual panel tells the story the numbers summarize: the TC fit is
flat at the percent level across four decades, while HB drifts systematically.
(Compare the previous notebook, where the ranking ran the other way — the model
choice belongs to the data, not to habit.)

## 💡 Takeaways

- **Two model keys, one comparison.** `rheofit.fit(df, "tc", …)` vs
  `rheofit.fit(df, "herschel_bulkley", …)` — the whole original `lmfit` +
  `pybroom` scaffold collapses into a few lines.
- **Identifiability is reported, not assumed.** Every parameter arrives with a
  relative uncertainty and the fit with a condition number — the TC fit here is
  tight on all three parameters (cond ≈ 15).
- **No universal winner.** TC crushes HB on this curve and loses to it on the
  Carbopol curve of the iterative-fitting notebook. Fit both; let RedChi² and
  the residuals decide.